# 02 Оценка физического успеха и проверка по видео

        Этот раздел решает ключевой вопрос: действительно ли success в логе означает, что робот взял чашку и поставил её на тарелку. Здесь старое геометрическое условие успеха и `physical_success` считаются отдельно, а успех и провал проверяются по ключевым кадрам видео.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: чтение примерных метрик этой темы


In [ ]:
snapshot_path = ASSET_DIR / "metrics_snapshot.json"
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
print(json.dumps(snapshot["best_summary"], ensure_ascii=False, indent=2))


## Checkpoint 2: что такое `physical_success`


In [ ]:
def physical_success(row, min_lift=0.03, min_lift_steps=3, upright_threshold=0.7):
    """Пример логики оценки. В реальном проекте ориентируйтесь на поля, которые записывает скрипт eval."""
    legacy = bool(row.get("legacy_success", row.get("success", False)))
    max_lift = float(row.get("max_target_lift", row.get("max_mug_lift", 0.0)))
    lifted_steps = int(row.get("lifted_steps", 0))
    upright = float(row.get("final_target_upright_cos", row.get("final_mug_upright_cos", 1.0)))
    return legacy and max_lift >= min_lift and lifted_steps >= min_lift_steps and upright >= upright_threshold


examples = [
    {"name": "геометрически успешно, но чашка опрокинута", "success": True, "max_mug_lift": 0.08, "lifted_steps": 80, "final_mug_upright_cos": 0.2},
    {"name": "толкнули к краю тарелки, но не схватили", "success": True, "max_mug_lift": 0.005, "lifted_steps": 0, "final_mug_upright_cos": 0.99},
    {"name": "настоящий захват и установка в вертикальном положении", "success": True, "max_mug_lift": 0.09, "lifted_steps": 120, "final_mug_upright_cos": 0.96},
]
md_table(["Пример", "physical_success"], [(e["name"], physical_success(e)) for e in examples])


Старый success можно использовать как вспомогательный показатель, но он не заменяет проверку по видео и по физическому состоянию. Особенно в задаче захвата чашки толчки, сдавливание и опрокидывание могут сделать конечную геометрию похожей на успех.


## Checkpoint 3: проверка ключевых кадров успеха и провала


In [ ]:
show_asset("smolvla_blue_failure_sequence.jpg", width=1100)
show_asset("smolvla_blue_success_sequence.jpg", width=1100)
show_asset("act_failure_sequence.jpg", width=1100)
show_asset("act_success_sequence.jpg", width=1100)


При просмотре видео следите по временной оси за четырьмя вещами: касается ли целевой чашки, устойчиво ли она схвачена, перенесена ли на тарелку и вертикальна ли в конечном состоянии.


## Checkpoint 4: шаблон пакетной статистики JSONL


In [ ]:
def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print(f"Файл не существует: {path}")
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(r.get("success") or r.get("legacy_success")) for r in rows)
    physical = sum(bool(r.get("physical_success")) for r in rows)
    return {"total": total, "legacy_success": legacy, "physical_success": physical}


example_jsonl = OUTPUT_ROOT / "eval_result.jsonl"
print("Положите сюда свой JSONL и запустите:", example_jsonl)
print(summarize_jsonl(example_jsonl))
